In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import shap

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
df1 = pd.read_csv('/home/u2021000941/Classification/first_output_file.csv')
df2 = pd.read_csv('/home/u2021000941/Classification/second_output_file.csv')

In [ ]:
combined_df = pd.concat([df1, df2])
grid_prod_df = pd.read_csv('/home/u2021000941/GridProd_1922_monthly.csv')
cleaned_df = combined_df.dropna()
cleaned_df = cleaned_df.drop('PM1_MEAN', axis=1)
cleaned_df = cleaned_df.sort_values(by=['IDCode', 'Year', 'Month'])

In [ ]:
cleaned_df = combined_df.drop('PM1_MEAN', axis=1)

In [ ]:
cleaned_df = cleaned_df.sort_values(by=['IDCode', 'Year', 'Month'])

In [ ]:
cleaned_df['Year'].unique()

In [ ]:
grid_info_df = pd.read_csv('/home/u2021000941/gridinfo_xy.csv')
temp_df = grid_prod_df[['IDCode', 'Year', 'Month', 'GridProd_Steel_tot', 'GridProd_Iron_tot']].copy()
merged_df = pd.merge(cleaned_df, temp_df, on=['IDCode', 'Year', 'Month'], how='left', indicator=True)
merged_df = pd.merge(merged_df, grid_info_df, on=['IDCode'], how='left')

In [ ]:
merged_df['y_binary'] = (merged_df['GridProd_Steel_tot'] > 0).astype(int)
merged_df = merged_df.drop(columns=['_merge', 'Polygon_ID', 'Centroid_Long', 'Centroid_Lat'])

In [ ]:
merged_df.info()

In [ ]:
merged_df['Year'].unique()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


pollutant_columns = ['CO_MEAN', 'NO2_MEAN', 'PM2_5_MEAN', 'PM10_MEAN', 'SO2_MEAN', 'LSTA_MEAN', 'LSTT_MEAN', 'NTL_MEAN', 'O3_MEAN']
titles = [
    'Comparison of CO Means',
    'Comparison of NO2 Means',
    'Comparison of PM2.5 Means',
    'Comparison of PM10 Means',
    'Comparison of SO2 Means',
    'Comparison of Land Surface Temperature (Day)',
    'Comparison of Land Surface Temperature (Night)',
    'Comparison of Nighttime Lights',
    'Comparison of O3 Means'
]


plt.figure(figsize=(18, 12))
for i, pollutant in enumerate(pollutant_columns):
    plt.subplot(3, 3, i + 1)
    
    grouped = merged_df.groupby(['Year', 'Month', 'y_binary'])[pollutant].mean().reset_index()
    
    pivoted = grouped.pivot(index=['Year', 'Month'], columns='y_binary', values=pollutant)
    pivoted = pivoted.rename(columns={0: 'Without Plant', 1: 'With Plant'})
    pivoted = pivoted.reset_index()
    pivoted['Date'] = pd.to_datetime(pivoted['Year'].astype(str) + '-' + pivoted['Month'].astype(str) + '-01')
    
    plt.plot(pivoted['Date'], pivoted['With Plant'], label='With Plant', marker='o')
    plt.plot(pivoted['Date'], pivoted['Without Plant'], label='Without Plant', marker='o')
    
    plt.title(titles[i])
    plt.xlabel('Year-Month')
    plt.ylabel(f'{pollutant}')
    plt.legend(loc='upper right')
    plt.xticks(pivoted['Date'][::5], pivoted['Date'].dt.strftime('%Y-%m')[::5], rotation=45)

plt.tight_layout()
plt.savefig('pollutant_comparison.pdf', format='pdf', dpi=800)
plt.show()


In [ ]:
X = merged_df.drop(['GridProd_Steel_tot', 'GridProd_Iron_tot', 'y_binary', 'Month'], axis=1)
y = merged_df['y_binary']

In [ ]:
imputer = SimpleImputer(strategy='mean')
X_imputed_array = imputer.fit_transform(X)

scaler = StandardScaler()
X_imputed_array = scaler.fit_transform(X_imputed_array)

X_imputed = pd.DataFrame(X_imputed_array, columns=X.columns)

In [ ]:
X = X_imputed

In [ ]:
X = X.drop('IDCode', axis=1)

# SMOTE Method

In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

In [ ]:
print(y_res.value_counts())

In [ ]:
X = X_res
y = y_res

In [ ]:
from sklearn.model_selection import train_test_split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y_res)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp)

In [ ]:
input_shape = (X_train.shape[1],)
model = Sequential()
model.add(Dense(units=64, input_shape=input_shape, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(units=32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(units=1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=11, batch_size=32)

In [ ]:
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.title('Loss over epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
model.summary()

# Model Performace Evaluation

In [ ]:
y_pred_prob = model.predict(X_test)

In [ ]:
y_pred = (y_pred_prob >= 0.5).astype(int)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6),dpi=1200)
cax = ax.matshow(confusion_matrix, cmap='Blues')
fig.colorbar(cax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_xticks(np.arange(2))
ax.set_yticks(np.arange(2))
ax.set_xticklabels(['No Steel Plant', 'Steel Plant'])
ax.set_yticklabels(['No Steel Plant', 'Steel Plant'])
for (i, j), val in np.ndenumerate(confusion_matrix):    
    ax.text(j, i, f'{val}', ha='center', va='center', color='black')
plt.title('Confusion Matrix Heatmap')
# plt.savefig(r'D:\Figures\Confusion Matrix Heatmap.pdf', format='pdf', bbox_inches='tight')
plt.show()

# ROC curve

In [ ]:
from sklearn.metrics import roc_curve, auc
y_score = model.predict(X_test).ravel() 

fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(dpi=1200)
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.savefig(r'D:\Figures\Receiver Operating Characteristic_v122.pdf', format='pdf', bbox_inches='tight')
plt.show()

# Precision-Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score


precision, recall, thresholds = precision_recall_curve(y_test, y_pred_prob)
average_precision = average_precision_score(y_test, y_pred_prob)


plt.figure(dpi=1200)
plt.plot(recall, precision, label=f'Average Precision = {average_precision:.2f}', color='blue')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.grid()
plt.savefig('Precision_Recall_Curve_v122.pdf', format='pdf', bbox_inches='tight')
plt.show()

# SHAP

In [ ]:
import shap
background = X_train.sample(n=10000, random_state=42) 
background_np = background.to_numpy()
X_explain_np = X_test[:10000].to_numpy()

In [ ]:
explainer = shap.Explainer(model, background_np)
shap_values_numpy = explainer.shap_values(X_explain_np)

In [ ]:
shap_values_raw = explainer(X_explain_np)

In [ ]:
feature_names = X_test.columns
shap_values_Explanation = shap.Explanation(values=shap_values_raw.values, 
                            base_values=shap_values_raw.base_values,
                                           data=X_explain_np, 
                                           feature_names=feature_names)  

# SHAP abstract figure

In [ ]:
plt.figure(figsize=(10, 5), dpi=1200)
shap.summary_plot(shap_values_Explanation, X_test[:10000], feature_names=feature_names, plot_type="dot", show=False)
plt.savefig("SHAP_Summary_Plot.pdf", format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5), dpi=1200)
shap.summary_plot(shap_values_numpy, X_test[:10000], plot_type="bar", show=False)
plt.title('SHAP_numpy Sorted Feature Importance')
plt.savefig("SHAP_numpy Sorted Feature Importance_122.pdf", format='pdf',bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import shap
from sklearn.utils import resample
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


n_simulations = 50
shap_values_list = []


input_shape = X.shape[1]
def create_model():
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_shape,)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

for i in range(n_simulations):
    print(f"Simulation {i + 1}/{n_simulations}")
    
    X_resampled, y_resampled = resample(X, y, replace=True, random_state=i)
    
    model = create_model()
    model.fit(X_resampled, y_resampled, epochs=10, batch_size=32, verbose=0)
    
    background = X.sample(n=100, random_state=42)  
    explainer = shap.KernelExplainer(model.predict, background)
    shap_values = explainer.shap_values(X)
    
    shap_values_list.append(np.abs(shap_values).mean(axis=0))

shap_values_df = pd.DataFrame(shap_values_list, columns=X.columns)

mean_shap = shap_values_df.mean()
lower_bound = shap_values_df.quantile(0.025)
upper_bound = shap_values_df.quantile(0.975)

plt.figure(figsize=(10, 5), dpi=1200)
plt.barh(X.columns, mean_shap, xerr=[mean_shap - lower_bound, upper_bound - mean_shap], align='center', color='blue', alpha=0.7)
plt.xlabel('Mean SHAP Value')
plt.ylabel('Features')
plt.title('SHAP Feature Importance with 50 Simulations (Neural Network)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("NN_SHAP_Feature_Importance_with_Simulations.pdf", format='pdf', bbox_inches='tight')
plt.show()
